In [1]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'rides'

In [2]:
import json
import importlib
import models              # this was the stale module
importlib.reload(models)   # re‑read the updated file

<module 'models' from '/workspaces/Machine-Learning-Zoomcamp-Homework/notebooks/models.py'>

In [3]:
from models import Ride, ride_deserializer

In [4]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    value_deserializer=ride_deserializer
)

In [5]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [ ]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    try:
        ride = message.value
        pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
        
        # Check if connection is still alive, reconnect if needed
        try:
            cur.execute("SELECT 1")
        except psycopg2.OperationalError:
            print("Connection lost, reconnecting...")
            cur.close()
            conn.close()
            conn = psycopg2.connect(
                host='localhost',
                port=5432,
                database='postgres',
                user='postgres',
                password='postgres'
            )
            conn.autocommit = True
            cur = conn.cursor()
        
        cur.execute(
            """INSERT INTO processed_events
               (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
               VALUES (%s, %s, %s, %s, %s)""",
            (ride.PULocationID, ride.DOLocationID,
             ride.trip_distance, ride.total_amount, pickup_dt)
        )
        count += 1
        if count % 100 == 0:
            print(f"Inserted {count} rows...")
    except Exception as e:
        print(f"Error processing message: {e}")
        continue

consumer.close()
cur.close()
conn.close()

Listening to rides and writing to PostgreSQL...


InterfaceError: cursor already closed

In [ ]:
import time
import psycopg2
from psycopg2 import OperationalError

def get_db_connection(max_retries=5, retry_delay=2):
    """Get database connection with retry logic"""
    for attempt in range(max_retries):
        try:
            conn = psycopg2.connect(
                host='localhost',
                port=5432,
                database='postgres',
                user='postgres',
                password='postgres'
            )
            conn.autocommit = True
            return conn
        except OperationalError as e:
            if attempt < max_retries - 1:
                print(f"Connection attempt {attempt + 1} failed: {e}")
                print(f"Retrying in {retry_delay} seconds...")
                time.sleep(retry_delay)
            else:
                print(f"Failed to connect after {max_retries} attempts")
                raise e

def ensure_table_exists(cur):
    """Ensure the processed_events table exists"""
    cur.execute("""
        CREATE TABLE IF NOT EXISTS processed_events (
            PULocationID INTEGER,
            DOLocationID INTEGER,
            trip_distance DOUBLE PRECISION,
            total_amount DOUBLE PRECISION,
            pickup_datetime TIMESTAMP
        )
    """)

# Initialize connection
conn = get_db_connection()
cur = conn.cursor()

# Ensure table exists
ensure_table_exists(cur)

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    try:
        ride = message.value
        pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)

        # Check if connection is still alive, reconnect if needed
        try:
            cur.execute("SELECT 1")
        except OperationalError:
            print("Connection lost, reconnecting...")
            cur.close()
            conn.close()
            conn = get_db_connection()
            cur = conn.cursor()
            ensure_table_exists(cur)

        cur.execute(
            """INSERT INTO processed_events
               (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
               VALUES (%s, %s, %s, %s, %s)""",
            (ride.PULocationID, ride.DOLocationID,
             ride.trip_distance, ride.total_amount, pickup_dt)
        )
        count += 1
        if count % 100 == 0:
            print(f"Inserted {count} rows...")

    except Exception as e:
        print(f"Error processing message: {e}")
        # Continue processing other messages
        continue

consumer.close()
cur.close()
conn.close()